In [51]:
# Library imports
import torch
import math
import random
import json
import nltk

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [2]:
# Determine what device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(device)

    print(f'CUDA on {device_name}')
else:
    print(f'CPU')

CUDA on NVIDIA RTX 500 Ada Generation Laptop GPU


In [3]:
# Check if everything's cleaned
allocated = torch.cuda.memory_allocated() / (1024 * 1024)
reserved = torch.cuda.memory_reserved() / (1024 * 1024)

print(f'Allocated: {allocated:.1f} MB; Reserved: {reserved:.1f} MB')

Allocated: 0.0 MB; Reserved: 0.0 MB


In [54]:
NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli' # https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
NLTK_MODEL = 'tokenizers/punkt/english.pickle'

In [59]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\erwin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)

Loading weights: 100%|██████████| 394/394 [00:00<00:00, 13250.55it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
dataset = json.load(open('datasets/ContraDoc/ContraDoc.json', 'r'))

print(f'Dataset loaded with {len(dataset["pos"])} positive and {len(dataset["neg"])} negative examples.')

Dataset loaded with 449 positive and 442 negative examples.


In [61]:
examples = list(dataset['pos'].values())

example = examples[random.randint(0, len(examples) - 1)]
sentences = nltk.tokenize.sent_tokenize(example['text'], language='english')

sentences

['Marseille, France The French prosecutor leading an investigation into the crash of Germanwings Flight 9525 insisted Wednesday that he was not aware of any video footage from on board the plane.',
 'Marseille prosecutor Brice Robin told CNN that "so far no videos were used in the crash investigation."',
 'He added, "A person who has such a video needs to immediately give it to the investigators."',
 "Robin's comments follow claims by two magazines, German daily Bild and French Paris Match, of a cell phone video showing the harrowing final seconds from on board Germanwings Flight 9525 as it crashed into the French Alps.",
 'All 150 on board were killed.',
 'Paris Match and Bild reported that the video was recovered from a phone at the wreckage site.',
 'The two publications described the supposed video, but did not post it on their websites.',
 'The publications said that they watched the video, which was found by a source close to the investigation.',
 '"One can hear cries of \'My God

In [ ]:
def predict(premise, hypothesis):
    input = tokenizer(premise, hypothesis, truncation=True, return_tensors="pt").to(device)

    output = model(input["input_ids"])
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]

    return {name: float(pred) for pred, name in zip(prediction, label_names)}

In [ ]:
premise = "The French had communicated to the British at Tellicherry that they would submit to any attempts to search their vessels."
hypothesis = "The French had communicated to the British at Tellicherry that they would not submit to any attempts to search their vessels , but Strachan and Cornwallis replied that they would enforce the blockade of Mangalore whatever the consequences."

print(predict(premise, hypothesis))

{'entailment': 0.2, 'neutral': 0.8, 'contradiction': 99.0}
